In [ ]:
# =====================================================================
# QwenCleo-ASR Long-Video GPU Worker (30s Chunking + Cloudflare Tunnel)
# =====================================================================
!pip install -q fastapi uvicorn httpx torch torchaudio transformers accelerate sentencepiece
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared

import asyncio, hashlib, hmac, json, logging, os, re, shutil, subprocess, threading, time
from pathlib import Path
import httpx, uvicorn
from fastapi import FastAPI, Request
import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("KaggleQwenCleoWorker")

app = FastAPI(title="QwenCleo ASR Worker")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float16"

logger.info(f"⏳ Loading QwenCleo-ASR model on {device} ({dtype})...")
from qwencleo_asr import QwenCleoASR
asr_model = QwenCleoASR(model_id="mohammedaly22/QwenCleo-ASR", device=device, dtype=dtype)
logger.info("✅ QwenCleo-ASR loaded successfully!")

def deduplicate_overlap_words(prev_text: str, curr_text: str, max_check: int = 8) -> str:
    if not prev_text or not curr_text:
        return curr_text
    prev_w = prev_text.strip().split()
    curr_w = curr_text.strip().split()
    max_k = min(len(prev_w), len(curr_w), max_check)
    for k in range(max_k, 0, -1):
        if " ".join(prev_w[-k:]).lower() == " ".join(curr_w[:k]).lower():
            return " ".join(curr_w[k:]).strip()
    return curr_text

async def process_job(data):
    job_id = data["job_id"]
    media_url = data["media_url"]
    cb_url = data["callback_url"]
    secret = data.get("callback_secret", "")
    
    work = Path(f"/tmp/job_{job_id}")
    work.mkdir(parents=True, exist_ok=True)
    vpath = work / "input.mp4"
    apath = work / "extracted.wav"
    
    try:
        # 1. Streaming Download
        logger.info(f"[{job_id}] 📥 Downloading media streaming...")
        async with httpx.AsyncClient(timeout=600.0) as client:
            async with client.stream("GET", media_url) as resp:
                resp.raise_for_status()
                with open(vpath, "wb") as f:
                    async for chunk in resp.aiter_bytes(chunk_size=1024*1024):
                        f.write(chunk)
        
        # 2. Extract 16kHz audio
        logger.info(f"[{job_id}] 🎵 Extracting 16kHz audio via FFmpeg...")
        subprocess.run(["ffmpeg", "-y", "-i", str(vpath), "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", str(apath)], check=True)
        probe = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(apath)], capture_output=True, text=True)
        total_duration = float(probe.stdout.strip() or 60.0)
        logger.info(f"[{job_id}] Total Duration: {total_duration:.1f}s ({total_duration/60:.1f} min)")

        # 3. 30-Second Chunking Loop (Fixes CUDA Out of Memory!)
        chunk_size = 30.0
        overlap = 2.0
        segments = []
        full_text_list = []
        prev_text = ""
        curr_start = 0.0
        seq = 1

        t0 = time.time()
        while curr_start < total_duration:
            curr_end = min(total_duration, curr_start + chunk_size)
            slice_dur = curr_end - curr_start
            slice_path = work / f"slice_{seq}.wav"
            
            # Slice audio chunk
            subprocess.run(["ffmpeg", "-y", "-ss", str(curr_start), "-i", str(apath), "-t", str(slice_dur), "-acodec", "copy", str(slice_path)], capture_output=True)
            
            # Transcribe chunk safely on GPU (Uses only ~500MB VRAM!)
            if slice_path.exists():
                res = asr_model.transcribe(str(slice_path))
                raw_text = res.text.strip()
                slice_path.unlink(missing_ok=True)
            else:
                raw_text = ""

            clean_text = deduplicate_overlap_words(prev_text, raw_text)
            if clean_text:
                segments.append({
                    "sequence": seq,
                    "start_time": round(curr_start, 2),
                    "end_time": round(curr_end, 2),
                    "text": clean_text
                })
                full_text_list.append(clean_text)
                prev_text = clean_text

            if seq % 10 == 0 or curr_end >= total_duration:
                pct = min(99, int((curr_end / total_duration) * 100))
                logger.info(f"[{job_id}] Progress: {pct}% ({curr_end/60:.1f} / {total_duration/60:.1f} min)")

            if curr_end >= total_duration:
                break
            curr_start = curr_end - overlap
            seq += 1

        full_text = " ".join(full_text_list).strip()
        elapsed = time.time() - t0
        rtf = elapsed / max(total_duration, 1.0)
        logger.info(f"[{job_id}] ⚡ ASR Done in {elapsed:.1f}s (RTF: {rtf:.3f}) - {len(segments)} segments")

        # 4. Authenticated HMAC Callback
        payload = {
            "status": "completed",
            "job_id": job_id,
            "video_id": data["video_id"],
            "lesson_id": data.get("lesson_id"),
            "course_id": data.get("course_id"),
            "duration": total_duration,
            "full_text": full_text,
            "segments": segments,
            "segment_count": len(segments),
            "rtf": round(rtf, 3)
        }
        body_bytes = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        timestamp_str = str(int(time.time()))
        sig_payload = f"{timestamp_str}.".encode("utf-8") + body_bytes
        sig = hmac.new(secret.encode("utf-8"), sig_payload, hashlib.sha256).hexdigest()

        headers = {
            "Content-Type": "application/json",
            "X-Job-ID": job_id,
            "X-Timestamp": timestamp_str,
            "X-Signature-SHA256": sig
        }

        logger.info(f"[{job_id}] 📤 Posting callback to LMS...")
        async with httpx.AsyncClient(timeout=60.0) as client:
            cb_res = await client.post(cb_url, content=body_bytes, headers=headers)
            logger.info(f"[{job_id}] LMS Callback Response: HTTP {cb_res.status_code}")

    except Exception as exc:
        logger.error(f"[{job_id}] Error in processing: {exc}")
    finally:
        shutil.rmtree(work, ignore_errors=True)
        torch.cuda.empty_cache()

@app.get("/health")
def health():
    return {"status": "healthy", "device": device, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}

@app.post("/jobs")
async def jobs(req: Request):
    b = await req.json()
    asyncio.create_task(process_job(b))
    return {"status": "accepted", "job_id": b.get("job_id")}

# Start Background Server
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
time.sleep(2)

# Start Cloudflare Tunnel
tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    if "trycloudflare.com" in line:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            print("\n" + "="*65)
            print(f"🚀 PUBLIC WORKER URL: {m.group(0)}")
            print("="*65 + "\n")
            break

print("🟢 Server is live and ready to receive transcription jobs!")
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("Stopping server...")

